# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deepikachebrolu/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from pathlib import Path
import json
import pandas as pd


repo = Path.cwd()

while repo != repo.parent and not (repo / "skills").exists():
    repo = repo.parent

print("Repository:", repo)


for filename in [
    "skills/building-baselines/SKILL.md",
    "skills/flyrank/flyrank-data/SKILL.md",
]:
    path = repo / filename

    print("\n" + "=" * 70)
    print(filename)
    print("=" * 70)

    if path.exists():
        print(path.read_text(encoding="utf-8"))
    else:
        print("File not found:", path)


csv_path = repo / "data" / "raw" / "content_refresh_anonymized.csv"

if csv_path.exists():
    df_sample = pd.read_csv(csv_path, nrows=5)

    print("\nStarter dataset columns:")
    print(df_sample.columns.tolist())

    print("\nDataset preview:")
    display(df_sample.head(3))
else:
    print("\nStarter dataset not found:", csv_path)

notebook_path = repo / "work" / "notebooks" / "w04_baseline_score.ipynb"

if notebook_path.exists():
    with open(notebook_path, encoding="utf-8") as f:
        notebook = json.load(f)

    print("\nML-07 notebook structure:")

    for i, cell in enumerate(notebook["cells"]):
        source = "".join(cell.get("source", []))
        if cell["cell_type"] == "markdown":
            print(f"\nMARKDOWN CELL {i}:\n{source[:500]}")
        else:
            print(f"\nCODE CELL {i}:\n{source[:200]}")
else:
    print("\nNotebook not found:", notebook_path)

Repository: c:\Users\Divya\Documents\flyrank-ml

skills/building-baselines/SKILL.md
---
name: building-baselines
description: Builds the transparent rule-based baseline every model must beat — a hand-written score with reason codes, ranked output, and precision@K evaluation. Use before training any model, or when someone reports model results with nothing to compare against.
---

# Building baselines

A model without a baseline is a number without a meaning. The baseline is a rule a human can
read — and its job is to be honestly beatable.

## Build it in this order

**1. Say the rule in plain words first.** "A page is worth reviewing if it used to get traffic,
it's getting old, and its position is slipping." If you can't say it, you can't code it.

**2. Code it as a transparent score.** Multiply/add simple conditions; no fitted weights:

```python
stale   = (df["days_since_update"] >= 180).astype(int)
visible = (df["impressions"] >= 500).astype(int)
df["score"] = stale * visible * df["

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0,good,page_3_5,down,-60.9



ML-07 notebook structure:

MARKDOWN CELL 0:
# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deepikachebrolu/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignme

MARKDOWN CELL 1:
## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

CODE CELL 2:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


MARKDOWN CELL 3:
## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

CODE CELL 4:
# Th

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule: Prioritize visible content with a potential CTR opportunity

I prioritize content with at least 500 impressions over the previous 90 days and an average search position between 1 and 20. Within that eligible set, I rank content by its impression volume multiplied by its CTR shortfall relative to the eligible-set median CTR.

The rule uses one reason code: `CTR_REVIEW`.

The action label is `REVIEW_TITLE_SNIPPET`.

The score is a decision-support heuristic, not a prediction of guaranteed additional clicks. It is intended to prioritize manual review.

### Signal verdicts

- **CTR versus position — MIXED:** CTR generally decreases at weaker positions, but the zero medians at positions 1–3 and 20+ show that position alone is not sufficient to identify opportunities.
- **Impression volume — CONFIRMED:** Higher-volume buckets have higher median CTR in this dataset. This is an association, not a causal result.

### Limitations

The starter dataset contains anonymized, content-level observations and 90-day aggregate metrics. It does not establish the causal impact of title or snippet changes. A high score indicates a review candidate, not a guaranteed improvement.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np


repo = Path.cwd()

while repo != repo.parent and not (repo / "data").exists():
    repo = repo.parent

data_path = repo / "data" / "raw" / "content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("Dataset loaded from:", data_path)


required = [
    "content_id",
    "client_id",
    "impressions_90d",
    "ctr",
    "avg_position",
]

missing = [col for col in required if col not in df.columns]

if missing:
    raise ValueError(f"Required columns are missing: {missing}")

for col in ["impressions_90d", "ctr", "avg_position"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")


eligible = df[
    df["impressions_90d"].notna()
    & df["ctr"].notna()
    & df["avg_position"].notna()
    & (df["impressions_90d"] > 0)
].copy()

print("Eligible content rows:", len(eligible))
print("Rows excluded because required signals are missing or impressions are zero:",
      len(df) - len(eligible))


Dataset shape: (30000, 44)
Dataset loaded from: c:\Users\Divya\Documents\flyrank-ml\data\raw\content_refresh_anonymized.csv
Eligible content rows: 30000
Rows excluded because required signals are missing or impressions are zero: 0


In [3]:
# Signal A: CTR bucket by average search position.
# CTR values in this dataset are stored as x100 percentages:
# 0.76 means 0.76%, not 76%.

eligible["position_bucket"] = pd.cut(
    eligible["avg_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["1-3", "4-10", "11-20", "20+"],
    right=True,
)

position_table = (
    eligible.groupby("position_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        median_ctr=("ctr", "median"),
        mean_ctr=("ctr", "mean"),
    )
    .reset_index()
)

display(position_table)

print(
    "Interpretation: CTR generally tends to be higher at stronger positions, "
    "but this table must be checked before assigning a verdict."
)

,position_bucket,n,median_ctr,mean_ctr
0,1-3,1141,0.00,2.714303
1,4-10,11842,0.16,0.651045
2,11-20,7273,0.10,0.323443
3,20+,8539,0.00,0.211333


Interpretation: CTR generally tends to be higher at stronger positions, but this table must be checked before assigning a verdict.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
import json

output_dir = repo / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

output_csv = output_dir / "baseline_action_score.csv"


queue = df[
    df["content_id"].notna()
    & df["impressions_90d"].notna()
    & df["ctr"].notna()
    & df["avg_position"].notna()
].copy()

queue["impressions_90d"] = pd.to_numeric(
    queue["impressions_90d"], errors="coerce"
)
queue["ctr"] = pd.to_numeric(queue["ctr"], errors="coerce")
queue["avg_position"] = pd.to_numeric(
    queue["avg_position"], errors="coerce"
)

queue = queue.dropna(
    subset=["impressions_90d", "ctr", "avg_position"]
)

queue = queue[
    (queue["impressions_90d"] >= 500)
    & (queue["avg_position"] >= 1)
    & (queue["avg_position"] <= 20)
].copy()

if queue.empty:
    raise ValueError("No eligible content found. Recheck the signal thresholds.")

reference_ctr = queue["ctr"].median()

queue["ctr_shortfall"] = (
    reference_ctr - queue["ctr"]
).clip(lower=0)

queue["score"] = (
    queue["impressions_90d"] * queue["ctr_shortfall"]
)

queue["reason_code"] = "CTR_REVIEW"
queue["action_label"] = "REVIEW_TITLE_SNIPPET"

queue = queue.sort_values(
    ["score", "impressions_90d", "content_id"],
    ascending=[False, False, True],
    kind="mergesort",
).reset_index(drop=True)

queue.insert(0, "rank", range(1, len(queue) + 1))

output_columns = [
    "rank",
    "content_id",
    "impressions_90d",
    "ctr",
    "avg_position",
    "ctr_shortfall",
    "score",
    "reason_code",
    "action_label",
]

queue[output_columns].to_csv(output_csv, index=False)

print("Eligible content:", len(queue))
print("Reference CTR:", reference_ctr)
print("Output written to:", output_csv)
print("\nTop 10 candidates:")

display(queue[output_columns].head(10))


Eligible content: 12009
Reference CTR: 0.21
Output written to: c:\Users\Divya\Documents\flyrank-ml\work\outputs\baseline_action_score.csv

Top 10 candidates:


,rank,content_id,impressions_90d,ctr,avg_position,ctr_shortfall,score,reason_code,action_label
0,1,content_8451fc6f034d,272144,0.03,2.3,0.18,48985.92,CTR_REVIEW,REVIEW_TITLE_SNIPPET
1,2,content_36ff89c8214e,295097,0.05,7.3,0.16,47215.52,CTR_REVIEW,REVIEW_TITLE_SNIPPET
2,3,content_c8e9d6ab9013,208678,0.00,9.7,0.21,43822.38,CTR_REVIEW,REVIEW_TITLE_SNIPPET
3,4,content_c84a0ab98e90,223271,0.03,7.8,0.18,40188.78,CTR_REVIEW,REVIEW_TITLE_SNIPPET
4,5,content_5fe46e04994d,517715,0.14,4.2,0.07,36240.05,CTR_REVIEW,REVIEW_TITLE_SNIPPET
5,6,content_8c19996aa890,509252,0.15,2.5,0.06,30555.12,CTR_REVIEW,REVIEW_TITLE_SNIPPET
6,7,content_453722754fea,140079,0.01,7.6,0.20,28015.80,CTR_REVIEW,REVIEW_TITLE_SNIPPET
7,8,content_4a6607efcb46,128068,0.01,2.2,0.20,25613.60,CTR_REVIEW,REVIEW_TITLE_SNIPPET
8,9,content_c1fe78bc4e37,134055,0.03,7.5,0.18,24129.90,CTR_REVIEW,REVIEW_TITLE_SNIPPET
9,10,content_91652435f57a,159590,0.06,7.8,0.15,23938.50,CTR_REVIEW,REVIEW_TITLE_SNIPPET


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:
top20 = queue.head(20).copy()

def review_action(row):
    return (
        f"Review title/snippet; observed CTR={row['ctr']:.3f}, "
        f"position={row['avg_position']:.1f}"
    )

def review_reason(row):
    return (
        f"{row['impressions_90d']:.0f} impressions and "
        f"CTR shortfall={row['ctr_shortfall']:.3f} "
        f"against reference CTR={reference_ctr:.3f}"
    )

def review_confidence(row):
    if row["impressions_90d"] >= 5000:
        return "Higher exposure; still requires manual review"
    return "Eligible exposure; verify query mix and context"

def review_what_could_be_wrong(row):
    return (
        "The query mix, search intent, SERP features, or position mix "
        "may explain the CTR; the aggregate data cannot prove that a "
        "title/snippet change will improve clicks."
    )

top20["review_action"] = top20.apply(review_action, axis=1)
top20["why_here"] = top20.apply(review_reason, axis=1)
top20["confidence_note"] = top20.apply(review_confidence, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(
    review_what_could_be_wrong, axis=1
)

review_columns = [
    "rank",
    "content_id",
    "score",
    "reason_code",
    "review_action",
    "why_here",
    "confidence_note",
    "what_would_make_it_wrong",
]

display(top20[review_columns])


,rank,content_id,score,reason_code,review_action,why_here,confidence_note,what_would_make_it_wrong
0,1,content_8451fc6f034d,48985.92,CTR_REVIEW,"Review title/snippet; observed CTR=0.030, posi...",272144 impressions and CTR shortfall=0.180 aga...,Higher exposure; still requires manual review,"The query mix, search intent, SERP features, o..."
1,2,content_36ff89c8214e,47215.52,CTR_REVIEW,"Review title/snippet; observed CTR=0.050, posi...",295097 impressions and CTR shortfall=0.160 aga...,Higher exposure; still requires manual review,"The query mix, search intent, SERP features, o..."
2,3,content_c8e9d6ab9013,43822.38,CTR_REVIEW,"Review title/snippet; observed CTR=0.000, posi...",208678 impressions and CTR shortfall=0.210 aga...,Higher exposure; still requires manual review,"The query mix, search intent, SERP features, o..."
3,4,content_c84a0ab98e90,40188.78,CTR_REVIEW,"Review title/snippet; observed CTR=0.030, posi...",223271 impressions and CTR shortfall=0.180 aga...,Higher exposure; still requires manual review,"The query mix, search intent, SERP features, o..."
4,5,content_5fe46e04994d,36240.05,CTR_REVIEW,"Review title/snippet; observed CTR=0.140, posi...",517715 impressions and CTR shortfall=0.070 aga...,Higher exposure; still requires manual review,"The query mix, search intent, SERP features, o..."
5,6,content_8c19996aa890,30555.12,CTR_REVIEW,"Review title/snippet; observed CTR=0.150, posi...",509252 impressions and CTR shortfall=0.060 aga...,Higher exposure; still requires manual review,"The query mix, search intent, SERP features, o..."
6,7,content_453722754fea,28015.80,CTR_REVIEW,"Review title/snippet; observed CTR=0.010, posi...",140079 impressions and CTR shortfall=0.200 aga...,Higher exposure; still requires manual review,"The query mix, search intent, SERP features, o..."
7,8,content_4a6607efcb46,25613.60,CTR_REVIEW,"Review title/snippet; observed CTR=0.010, posi...",128068 impressions and CTR shortfall=0.200 aga...,Higher exposure; still requires manual review,"The query mix, search intent, SERP features, o..."
8,9,content_c1fe78bc4e37,24129.90,CTR_REVIEW,"Review title/snippet; observed CTR=0.030, posi...",134055 impressions and CTR shortfall=0.180 aga...,Higher exposure; still requires manual review,"The query mix, search intent, SERP features, o..."
9,10,content_91652435f57a,23938.50,CTR_REVIEW,"Review title/snippet; observed CTR=0.060, posi...",159590 impressions and CTR shortfall=0.150 aga...,Higher exposure; still requires manual review,"The query mix, search intent, SERP features, o..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [12]:
# Inspect high-ranked items that may be weak recommendations.

weak_picks = queue[
    (queue["score"] > 0)
    & (
        (queue["ctr"] >= reference_ctr)
        | (queue["avg_position"] > 10)
    )
].sort_values(
    ["score", "impressions_90d"],
    ascending=[False, False],
).head(5)

print("Potential weak picks for manual review:")

if weak_picks.empty:
    print("No weak picks matched these diagnostic conditions.")
else:
    display(
        weak_picks[
            [
                "rank",
                "content_id",
                "impressions_90d",
                "ctr",
                "avg_position",
                "ctr_shortfall",
                "score",
            ]
        ]
    )

# Leakage check: confirm that no outcome-derived fields are used.
forbidden_columns = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
}

scoring_inputs = {
    "impressions_90d",
    "ctr",
    "avg_position",
}

assert scoring_inputs.isdisjoint(forbidden_columns)

score_inputs = {
    "impressions_90d",
    "ctr_shortfall",
}

assert score_inputs.isdisjoint(forbidden_columns)
assert not (forbidden_columns & set(output_columns))

print("\nLeakage check passed for the scoring inputs.")
print("Label-derived fields are excluded from the exported queue.")
print("The score uses only observed impressions and CTR shortfall.")
print("Future-window and label-derived fields are not used by this rule.")

Potential weak picks for manual review:


,rank,content_id,impressions_90d,ctr,avg_position,ctr_shortfall,score
58,59,content_bd765ab754cf,71283,0.08,17.2,0.13,9266.79
68,69,content_42634cb0c5a3,43175,0.02,18.8,0.19,8203.25
77,78,content_00202ac57009,61832,0.09,18.0,0.12,7419.84
103,104,content_8223440cd40c,35051,0.03,17.9,0.18,6309.18
107,108,content_617c14bb6cfa,41670,0.06,11.2,0.15,6250.50



Leakage check passed for the scoring inputs.
Label-derived fields are excluded from the exported queue.
The score uses only observed impressions and CTR shortfall.
Future-window and label-derived fields are not used by this rule.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### ML-07 completion notes

- Two signals were examined with bucket tables and observation counts.
- CTR versus position was classified as MIXED.
- Impression volume was classified as CONFIRMED, with an association-versus-causation caveat.
- One transparent rule produces a numeric score, one reason code, and one action label.
- The ranked queue is generated at `work/outputs/baseline_action_score.csv`.
- The top ten have individual explanations and counterarguments.
- Weak picks are inspected separately.
- Scoring inputs exclude label-derived fields and future-window outcomes.

The score is a rule-based prioritization baseline. It does not estimate causal uplift or guarantee that recommended changes will increase clicks.